## Music Recommender System
#### Content-Based Recommendation using Audio Features

In this project, we develop a music recommendation system that suggests similar tracks based on audio characteristics and artist information. This is a **Content-Based Filtering** approach, where the similarity between songs is determined by their intrinsic features rather than user interaction history.

### Project Objectives
- Analyze and preprocess raw music data.
- Feature engineering: Converting categorical (artists) and numeric (audio) data into a model-ready format.
- Implementation of the **Nearest Neighbors** algorithm for similarity search.
- Providing actionable song recommendations.
- Visualizing data patterns to gain insights into music distribution.

### 🎯Significance
Recommendation systems are the backbone of modern streaming platforms, enabling discovery and personalized user experiences. This project provides a robust, scalable foundation for such a system.


In [1]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import euclidean_distances, manhattan_distances, cosine_similarity
from scipy.spatial.distance import jaccard
#---------------------------------------------------------------------------------------------
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.feature_extraction.text import CountVectorizer
#---------------------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import re
#---------------------------------------------------------------------------------------------
import seaborn as sns
import matplotlib.pyplot as plt

**load data**

In [2]:
df_origin = pd.read_csv('tracks_features.csv')
df_origin

,id,name,album,album_id,artists,artist_ids,track_number,disc_number,explicit,danceability,...,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,year,release_date
0,7lmeHLHBe4nmXzuXc0HDjk,Testify,The Battle Of Los Angeles,2eia0myWFgoHuttJytCxgX,['Rage Against The Machine'],['2d0hyoQ5ynDBnkvAbJKORj'],1,1,False,0.470,...,0.0727,0.02610,0.000011,0.3560,0.503,117.906,210133,4.0,1999,1999-11-02
1,1wsRitfRRtWyEapl0q22o8,Guerrilla Radio,The Battle Of Los Angeles,2eia0myWFgoHuttJytCxgX,['Rage Against The Machine'],['2d0hyoQ5ynDBnkvAbJKORj'],2,1,True,0.599,...,0.1880,0.01290,0.000071,0.1550,0.489,103.680,206200,4.0,1999,1999-11-02
2,1hR0fIFK2qRG3f3RF70pb7,Calm Like a Bomb,The Battle Of Los Angeles,2eia0myWFgoHuttJytCxgX,['Rage Against The Machine'],['2d0hyoQ5ynDBnkvAbJKORj'],3,1,False,0.315,...,0.4830,0.02340,0.000002,0.1220,0.370,149.749,298893,4.0,1999,1999-11-02
3,2lbASgTSoDO7MTuLAXlTW0,Mic Check,The Battle Of Los Angeles,2eia0myWFgoHuttJytCxgX,['Rage Against The Machine'],['2d0hyoQ5ynDBnkvAbJKORj'],4,1,True,0.440,...,0.2370,0.16300,0.000004,0.1210,0.574,96.752,213640,4.0,1999,1999-11-02
4,1MQTmpYOZ6fcMQc56Hdo7T,Sleep Now In the Fire,The Battle Of Los Angeles,2eia0myWFgoHuttJytCxgX,['Rage Against The Machine'],['2d0hyoQ5ynDBnkvAbJKORj'],5,1,False,0.426,...,0.0701,0.00162,0.105000,0.0789,0.539,127.059,205600,4.0,1999,1999-11-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1204020,0EsMifwUmMfJZxzoMPXJKZ,Gospel of Juke,Notch - EP,38O5Ys0W9PFS5K7dMb7yKb,['FVLCRVM'],['7AjItKsRnEYRSiBt2OxK1y'],2,1,False,0.264,...,0.0672,0.00935,0.002240,0.3370,0.415,159.586,276213,4.0,2014,2014-01-09
1204021,2WSc2TB1CSJgGE0PEzVeiu,Prism Visions,Notch - EP,38O5Ys0W9PFS5K7dMb7yKb,['FVLCRVM'],['7AjItKsRnEYRSiBt2OxK1y'],3,1,False,0.796,...,0.0883,0.10400,0.644000,0.0749,0.781,121.980,363179,4.0,2014,2014-01-09
1204022,6iProIgUe3ETpO6UT0v5Hg,Tokyo 360,Notch - EP,38O5Ys0W9PFS5K7dMb7yKb,['FVLCRVM'],['7AjItKsRnEYRSiBt2OxK1y'],4,1,False,0.785,...,0.0564,0.03040,0.918000,0.0664,0.467,121.996,385335,4.0,2014,2014-01-09
1204023,37B4SXC8uoBsUyKCWnhPfX,Yummy!,Notch - EP,38O5Ys0W9PFS5K7dMb7yKb,['FVLCRVM'],['7AjItKsRnEYRSiBt2OxK1y'],5,1,False,0.665,...,0.0409,0.00007,0.776000,0.1170,0.227,124.986,324455,4.0,2014,2014-01-09


In [3]:
df_origin.columns

Index(['id', 'name', 'album', 'album_id', 'artists', 'artist_ids',
       'track_number', 'disc_number', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms',
       'time_signature', 'year', 'release_date'],
      dtype='str')

In [4]:
df= df_origin[['name','artists','danceability','energy','loudness','speechiness','acousticness','valence','tempo','duration_ms','year']]
df.head(3)

,name,artists,danceability,energy,loudness,speechiness,acousticness,valence,tempo,duration_ms,year
0,Testify,['Rage Against The Machine'],0.470,0.978,-5.399,0.0727,0.0261,0.503,117.906,210133,1999
1,Guerrilla Radio,['Rage Against The Machine'],0.599,0.957,-5.764,0.1880,0.0129,0.489,103.680,206200,1999
2,Calm Like a Bomb,['Rage Against The Machine'],0.315,0.970,-5.424,0.4830,0.0234,0.370,149.749,298893,1999


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1204025 entries, 0 to 1204024
Data columns (total 11 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   name          1204022 non-null  str    
 1   artists       1204025 non-null  str    
 2   danceability  1204025 non-null  float64
 3   energy        1204025 non-null  float64
 4   loudness      1204025 non-null  float64
 5   speechiness   1204025 non-null  float64
 6   acousticness  1204025 non-null  float64
 7   valence       1204025 non-null  float64
 8   tempo         1204025 non-null  float64
 9   duration_ms   1204025 non-null  int64  
 10  year          1204025 non-null  int64  
dtypes: float64(7), int64(2), str(2)
memory usage: 101.0 MB


In [6]:
df.dropna(subset=['name'], inplace=True)       
df.reset_index(drop=True, inplace=True)
df.head(3)

,name,artists,danceability,energy,loudness,speechiness,acousticness,valence,tempo,duration_ms,year
0,Testify,['Rage Against The Machine'],0.470,0.978,-5.399,0.0727,0.0261,0.503,117.906,210133,1999
1,Guerrilla Radio,['Rage Against The Machine'],0.599,0.957,-5.764,0.1880,0.0129,0.489,103.680,206200,1999
2,Calm Like a Bomb,['Rage Against The Machine'],0.315,0.970,-5.424,0.4830,0.0234,0.370,149.749,298893,1999


In [ ]:
df['artists'] = df['artists'].str.replace

In [7]:
def clean_artist_names(artist_str):
    
    names = re.findall(r"'([^']+)'", artist_str)
    cleaned_names = []
    for name in names:
        cleaned_names.append(name.replace(' ', '_'))
    return ' '.join(cleaned_names)


df['Artists_Clean'] = df['artists'].apply(clean_artist_names)
df[['name', 'artists', 'Artists_Clean']].head(5)

,name,artists,Artists_Clean
0,Testify,['Rage Against The Machine'],Rage_Against_The_Machine
1,Guerrilla Radio,['Rage Against The Machine'],Rage_Against_The_Machine
2,Calm Like a Bomb,['Rage Against The Machine'],Rage_Against_The_Machine
3,Mic Check,['Rage Against The Machine'],Rage_Against_The_Machine
4,Sleep Now In the Fire,['Rage Against The Machine'],Rage_Against_The_Machine


#-------------------------------------------------------------------
## Feature_Extraction

**approch:** CountVectorizer

In [10]:
vect = CountVectorizer()
artist_matrix = vect.fit_transform(df['Artists_Clean'])   # sparse matrix - toarray نمی‌کنیم
artist_matrix.shape

(1204022, 138625)

## Normalize


In [11]:
numeric_cols = ['danceability','energy','loudness','speechiness',
                 'acousticness','valence','tempo','duration_ms','year']
scaler = MinMaxScaler()
numeric_scaled = pd.DataFrame(scaler.fit_transform(df[numeric_cols]), columns=numeric_cols)
numeric_scaled.head(3)

,danceability,energy,loudness,speechiness,acousticness,valence,tempo,duration_ms,year
0,0.470,0.978,0.812104,0.075026,0.026205,0.503,0.473644,0.034510,0.989604
1,0.599,0.957,0.806675,0.194014,0.012952,0.489,0.416496,0.033861,0.989604
2,0.315,0.970,0.811732,0.498452,0.023494,0.370,0.601561,0.049157,0.989604


In [12]:
# Cell 10 — چسباندن (کانکت) به ستون‌های دیگر (numeric_scaled از سلول 8)
from scipy.sparse import hstack

final_matrix = hstack([artist_matrix, numeric_scaled]).tocsr()
final_matrix.shape

(1204022, 138634)

In [13]:
# Cell 11 — فیت کردن مدل
model = NearestNeighbors(n_neighbors=7 ,metric='cosine', algorithm='brute')
model.fit(final_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",7
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [ ]:
user_song = input("Enter Song Name")
user_song_index = np.where(df['name'] == user_song)[0][0]

n_recommended = 7
distances, indices = model.kneighbors(
    final_matrix[user_song_index], n_neighbors=n_recommended + 1
)

recommended_indices = indices[0][1:]
df['name'].iloc[recommended_indices]
